# paramz

## Example 1: a model with an objective and its gradient, optimized

In [ ]:
# Example 1: a quadratic with its minimum at 3; optimize() finds it through scipy
import numpy as np
import paramz

class Quadratic(paramz.Model):
    def __init__(self):
        super().__init__(name="quadratic")
        self.x = paramz.Param("x", np.array([10.0]))
        self.link_parameter(self.x)
    def objective_function(self):
        return float(((self.x - 3.0) ** 2).sum())
    def parameters_changed(self):
        self.x.gradient = 2.0 * (self.x - 3.0)

model = Quadratic()
model.optimize(messages=False)
assert abs(float(model.x[0]) - 3.0) < 1e-4, model.x
print("Example 1: x ->", float(model.x[0]), "objective", model.objective_function())

## Example 2: a positive constraint keeps a parameter above zero

In [ ]:
# Example 2: constrain_positive optimises in a transformed space (Logexp); the value never goes negative
import numpy as np
import paramz

class Pull(paramz.Model):
    def __init__(self):
        super().__init__(name="pull")
        self.s = paramz.Param("s", np.array([2.0]))
        self.link_parameter(self.s)
        self.s.constrain_positive(warning=False)
    def objective_function(self):
        return float(((self.s + 1.0) ** 2).sum())  # unconstrained minimum at -1
    def parameters_changed(self):
        self.s.gradient = 2.0 * (self.s + 1.0)

m = Pull()
m.optimize(messages=False)
assert float(m.s[0]) > 0 and float(m.s[0]) < 1e-2, m.s
print("Example 2: s stayed positive:", float(m.s[0]), "constraints:", list(m.constraints.properties()))

## Example 3: fix a parameter and optimize the rest

In [ ]:
# Example 3: a fixed parameter is left out of the optimiser's array
import numpy as np
import paramz

class Two(paramz.Model):
    def __init__(self):
        super().__init__(name="two")
        self.a = paramz.Param("a", np.array([5.0]))
        self.b = paramz.Param("b", np.array([5.0]))
        self.link_parameters(self.a, self.b)
    def objective_function(self):
        return float(((self.a - 1) ** 2 + (self.b - 2) ** 2).sum())
    def parameters_changed(self):
        self.a.gradient = 2 * (self.a - 1); self.b.gradient = 2 * (self.b - 2)

m = Two(); m.a.fix()
assert m.optimizer_array.size == 1
m.optimize(messages=False)
assert float(m.a[0]) == 5.0 and abs(float(m.b[0]) - 2.0) < 1e-4
print("Example 3: a fixed at", float(m.a[0]), "b ->", round(float(m.b[0]), 6))

## Example 4: the gradient checker catches a wrong gradient

In [ ]:
# Example 4: checkgrad compares the analytic gradient with finite differences
import numpy as np
import paramz

class Cubic(paramz.Model):
    def __init__(self, wrong=False):
        super().__init__(name="cubic"); self.wrong = wrong
        self.w = paramz.Param("w", np.array([0.7, -1.3])); self.link_parameter(self.w)
    def objective_function(self):
        return float((self.w ** 3).sum())
    def parameters_changed(self):
        self.w.gradient = (2.0 if self.wrong else 3.0) * self.w ** 2

assert bool(Cubic().checkgrad()) is True
assert bool(Cubic(wrong=True).checkgrad()) is False
print("Example 4: right gradient passes, wrong gradient fails")

## Example 5: nested Parameterized objects and parameter lookup

In [ ]:
# Example 5: parameters of linked children are reachable by name and counted by the parent
import numpy as np
import paramz

kernel = paramz.Parameterized("kernel")
kernel.variance = paramz.Param("variance", 1.5); kernel.lengthscale = paramz.Param("lengthscale", np.ones(3))
kernel.link_parameters(kernel.variance, kernel.lengthscale)
root = paramz.Parameterized("root"); root.noise = paramz.Param("noise", 0.1)
root.link_parameters(kernel, root.noise)
assert root.size == 5 and root.num_params == 2
assert np.allclose(root.param_array, [1.5, 1, 1, 1, 0.1])
assert root.kernel.lengthscale.size == 3 and float(root.noise[0]) == 0.1
print("Example 5:", root.size, "values;", [p.hierarchy_name() for p in root.flattened_parameters])